# Representative Experimental Template

This notebook is a representative template for one experimental variant reported in the paper.

The complete experimental campaign was executed across several Kaggle and Colab sessions due to GPU and runtime constraints. This notebook documents the execution flow used to generate the final JSON result files.

The frozen JSON outputs used in the paper are provided in the repository, so reviewers can verify the reported metrics without re-running all notebooks.

---
**Solver note:** For PrOntoQA and ProofWriter, the symbolic solver is SWI-Prolog. For FOLIO, the symbolic solver is Prover9. The notebook templates show the general execution structure, while the final FOLIO outputs included in the repository were generated with the Prover9 configuration.


In [ ]:
!apt-get install -y swi-prolog -q
!pip install transformers accelerate datasets -q

In [ ]:
import os, gc, json, time, re, torch, subprocess, tempfile, math
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secret   = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
MODEL_NAME   = "Qwen/Qwen2.5-7B-Instruct"   # change: unsloth/Meta-Llama-3.1-8B-Instruct | google/gemma-2-9b-it
DATASET_NAME = "proofwriter"                  # change: folio | proofwriter | prontoqa
MAX_NEW_TOK  = 512
MAX_REFINE   = 3
BEAM_K       = 5         # used only in beam notebooks
LENGTH_PEN   = 1.0       # used only in beam notebooks
OUTPUT_PATH  = f"/kaggle/working/poss_logiclm_{DATASET_NAME}_results.json"
print("Config OK")

In [ ]:
# ── LOAD DATASET ──────────────────────────────────────────────────────────────
# Uncomment the right block for your dataset

# --- FOLIO ---
# ds      = load_dataset("yale-nlp/FOLIO", split="validation", token=hf_token)
# dataset = list(ds)

# --- ProofWriter ---
ds      = load_dataset("tasksource/proofwriter", split="test", token=hf_token)
dataset = list(ds)

# --- PrOntoQA ---
# ds      = load_dataset("renma/ProntoQA", token=hf_token)
# dataset = list(ds["validation"])

print(f"Loaded {len(dataset)} examples")

In [ ]:
def format_example(ex):
    # ProofWriter
    context    = ex.get("theory", ex.get("context", ""))
    conclusion = ex.get("question", ex.get("conclusion", ""))
    label      = str(ex.get("label", "")).strip().lower()
    if label in ("true","1"):   label = "True"
    elif label in ("false","0"): label = "False"
    else:                         label = "Unknown"
    return context, conclusion, label

    # FOLIO — uncomment if using FOLIO
    # premises   = ex.get("premises", [])
    # context    = " ".join(premises) if isinstance(premises, list) else premises
    # conclusion = ex.get("conclusion", "")
    # label      = str(ex.get("label","")).lower()
    # if label in ("true","1"): label = "True"
    # elif label in ("false","0"): label = "False"
    # else: label = "Uncertain"
    # return context, conclusion, label

    # PrOntoQA — uncomment if using PrOntoQA
    # context    = ex.get("context", "")
    # conclusion = ex.get("question", "")
    # label      = str(ex.get("answer","")).strip()
    # if label.upper() == "A": label = "True"
    # elif label.upper() == "B": label = "False"
    # return context, conclusion, label

print("format_example OK")

In [ ]:
import subprocess, tempfile, os, re

def norm(s):
    s = str(s).lower().strip().replace("the ", "")
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

def convert_atom(a):
    a = a.strip().rstrip(".").replace("()", "")
    a = a.replace("NOT ", "not ").replace("not ", "not ")
    neg = a.lower().startswith("not ")
    if neg: a = a[4:].strip()
    m = re.match(r"(\w+)\((.*)\)", a)
    if not m: return None
    pred = norm(m.group(1))
    args = [x.upper() if x.strip().lower() in ["x","y","z"] else norm(x.strip())
            for x in m.group(2).split(",")]
    atom = f"{pred}({','.join(args)})"
    return f"neg_{atom}" if neg else atom

def fol_to_prolog(formalization):
    lines = []; in_prem = False
    for raw in formalization.split("\n"):
        l = raw.strip()
        if not l: continue
        if "PREMISES" in l.upper(): in_prem = True; continue
        if "CONCLUSION" in l.upper() or "PREDICATES" in l.upper(): in_prem = False; continue
        if not in_prem: continue
        l = l.rstrip(".")
        l = re.sub(r"^forall\s+\w+\s*", "", l).strip()
        if l.startswith("(") and l.endswith(")"): l = l[1:-1].strip()
        if "->" in l:
            parts = [p.strip() for p in l.split("->")]
            head = convert_atom(parts[-1])
            body = [convert_atom(x) for x in parts[:-1]]
            if head and all(body): lines.append(f"{head} :- {', '.join(body)}.")
        else:
            atom = convert_atom(l)
            if atom: lines.append(f"{atom}.")
    return "\n".join(lines)

def get_query(conclusion):
    s = conclusion.lower().strip().rstrip(".")
    if '?' in s: s = s.split('?',1)[1].strip()
    s = re.sub(r"\s+", " ", s.replace("does not","not").replace("do not","not"))
    m = re.match(r"^(?:the\s+)?(.+?)\s+is\s+not\s+(.+)$", s)
    if m: return f"neg_{norm(m.group(2))}({norm(m.group(1))})"
    m = re.match(r"^(?:the\s+)?(.+?)\s+is\s+(?:a\s+|an\s+)?(.+)$", s)
    if m: return f"{norm(m.group(2))}({norm(m.group(1))})"
    m = re.match(r"^(?:the\s+)?(.+?)\s+(not\s+)?(\w+)\s+(?:the\s+)?(.+)$", s)
    if m:
        subj=norm(m.group(1)); neg=m.group(2); verb=norm(m.group(3)); obj=norm(m.group(4))
        return f"{'neg_' if neg else ''}{verb}({subj},{obj})"
    return ""

def run_prolog(program, query):
    try:
        if not query: return "Unknown"
        preds = set(re.findall(r"(\w+)\(", program))
        discontig = "\n".join(f":- discontiguous {p}/1.\n:- discontiguous {p}/2." for p in preds)
        pos = query[4:] if query.startswith("neg_") else query
        neg = query if query.startswith("neg_") else f"neg_{query}"
        goal = f"""
:- (
    (catch(({pos}), _, fail), catch(({neg}), _, fail) -> write(conflict));
    (catch(({pos}), _, fail) -> write(true));
    (catch(({neg}), _, fail) -> write(false));
    write(unknown)
), halt.
"""
        with tempfile.NamedTemporaryFile(mode="w", suffix=".pl", delete=False) as f:
            f.write(discontig + "\n" + program + "\n" + goal); fname = f.name
        res = subprocess.run(["swipl", "-q", "-f", fname], capture_output=True, text=True, timeout=10)
        os.unlink(fname)
        out = res.stdout.lower()
        if "true" in out: return "True"
        if "false" in out: return "False"
        return "Unknown"
    except: return "Unknown"

def solve(formalization, conclusion):
    query   = get_query(conclusion)
    program = fol_to_prolog(formalization)
    return run_prolog(program, query), query

print("Solver OK")

In [ ]:
def pbs_pipeline(context, conclusion, formalisations, log_scores):
    K     = len(formalisations)
    max_s = max(log_scores)
    poss  = [math.exp(s - max_s) for s in log_scores]

    labels = []
    for fi in formalisations:
        pred_i, _ = solve(fi, conclusion)
        labels.append(pred_i)

    # Possibilistic aggregation
    pi = {"True": 0.0, "False": 0.0, "Unknown": 0.0}
    W  = {"True": 0.0, "False": 0.0, "Unknown": 0.0}
    for i, l in enumerate(labels):
        pi[l] = max(pi[l], poss[i])
        W[l] += poss[i]

    # Necessity
    pi_max = max(pi["True"], pi["False"], pi["Unknown"])
    N = {}
    for l in ("True", "False", "Unknown"):
        others = [pi[k] for k in pi if k != l]
        N[l] = 1.0 - max(others) if others else 1.0

    if W["True"] > 0 or W["False"] > 0:
        decision = max(["True", "False"], key=lambda x: (W[x], pi[x]))
    else:
        decision = "Unknown"

    return decision, {
        "solver_labels": labels,
        "poss":  [round(p, 6) for p in poss],
        "pi":    {k: round(v, 6) for k, v in pi.items()},
        "W":     {k: round(v, 6) for k, v in W.items()},
        "N":     {k: round(v, 6) for k, v in N.items()},
    }

print("pbs_pipeline OK")

In [ ]:
# PBS uses pre-generated beam database
# Upload your beam database as Kaggle dataset input
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Replace path with the one shown above
with open("/kaggle/input/your-db/beam_database.json") as f:
    db = json.load(f)

correct = 0; results = []
total   = len(db)
for i, item in enumerate(db):
    conclusion = item["conclusion"]
    gold       = item["label"]
    pred, info = pbs_pipeline(item["context"], conclusion,
                               item["formalisations"], item["log_scores"])
    ok = pred == gold; correct += ok
    results.append({"id": item.get("id", i), "gold": gold, "pred": pred,
                    "correct": ok, **info})
    if (i+1) % 50 == 0:
        print(f"{i+1}/{total} | acc={correct/(i+1)*100:.2f}%")

acc      = correct / total * 100
exe_rate = sum(1 for r in results if r["pred"] != "Unknown") / total * 100
print("="*50)
print(f"Accuracy : {acc:.2f}%  |  Exe Rate : {exe_rate:.2f}%")
print("="*50)
with open(OUTPUT_PATH, "w") as f:
    json.dump({"model": MODEL_NAME, "dataset": DATASET_NAME, "method": "PBS",
               "stats": {"total": total, "correct": correct,
                         "accuracy": round(acc,2), "exe_rate": round(exe_rate,2)},
               "results": results}, f, indent=2)
print("Saved:", OUTPUT_PATH)